# Capítulos 22 a 25 · Laboratório dos futuros: WIN e WDO no 5 minutos
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados: CSVs exportados de uma plataforma de negociação (série contínua do WIN e do WDO, 5 minutos), de 24/09 a 13/10/2026 (exemplos) e de 02/01 a 14/10/2026 (testes das Partes 6 e 7). Os arquivos estão na pasta `dados_futuros` do repositório do livro, e o notebook os baixa sozinho no Colab. Para usar os seus, exporte da sua plataforma no mesmo formato e ponha na pasta `dados_futuros`. A mesma pasta traz dois anos de WIN e WDO (02/09/2024 a 02/09/2026) para você repetir os testes numa amostra maior; nessa série contínua, os preços anteriores a cada rolagem vêm ajustados por um fator. As Partes 2 a 5 mostram os conceitos no gráfico, com poucos pregões; as Partes 6 e 7 testam as regras em 196 pregões, ainda uma amostra curta (veja o erro padrão). O volume é o número de contratos, espalhado por igual entre a máxima e a mínima de cada barra (aproximação do perfil feito com negócios, tick a tick).

In [ ]:
!pip install -q yfinance

In [ ]:
import glob
import os
import numpy as np
import pandas as pd

PASTA = "dados_futuros"
ARQ = {"WIN": "win_5min_2026-09-24_a_2026-10-13.csv", "WDO": "wdo_5min_2026-09-24_a_2026-10-06.csv"}
LONGO = {"WIN": "win_5min_2026-01-02_a_2026-10-14.csv", "WDO": "wdo_5min_2026-01-02_a_2026-10-14.csv"}
URL_DADOS = "https://raw.githubusercontent.com/traderbrasil/livro-analise-tecnica/v4.0/dados_futuros/"
os.makedirs(PASTA, exist_ok=True)
for nome in list(ARQ.values()) + list(LONGO.values()):   # no Colab, baixa os arquivos do repositório do livro
    if not os.path.exists(os.path.join(PASTA, nome)):
        try:
            import urllib.request
            urllib.request.urlretrieve(URL_DADOS + nome, os.path.join(PASTA, nome))
        except Exception as erro:
            print("Não consegui baixar", nome, erro)
TICK = {"WIN": 5.0, "WDO": 0.5}
PASSO = {"WIN": 50.0, "WDO": 1.0}        # tamanho da faixa de preço do perfil (WIN: 10 ticks; WDO: 2 ticks)
nn = lambda x: f"{x:,.0f}".replace(",", ".")
br = lambda x, k=1: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")

## Parte 1: dados

In [ ]:
def carrega(arquivo):
    d = pd.read_csv(arquivo, sep=";", header=None, decimal=",")
    d.columns = ["ativo", "dia", "hora", "Open", "High", "Low", "Close", "fin", "Volume"]
    d.index = pd.to_datetime(d["dia"] + " " + d["hora"], format="%d/%m/%Y %H:%M:%S")
    return d.sort_index()[["Open", "High", "Low", "Close", "Volume"]]


faltam = [a for a in list(ARQ.values()) + list(LONGO.values()) if not os.path.exists(os.path.join(PASTA, a))]
if faltam:
    raise SystemExit(f"Coloque os CSVs exportados na pasta '{PASTA}': {faltam}. Achei: {glob.glob(PASTA + '/*.csv')}")
DADOS = {k: carrega(os.path.join(PASTA, a)) for k, a in ARQ.items()}

print("Parte 1. Dados exportados da plataforma, barras de 5 minutos")
for k, d in DADOS.items():
    fora = ((d[["Open", "High", "Low", "Close"]] * 2) % 1 != 0).any(axis=1) if k == "WDO" else \
        (d[["Open", "High", "Low", "Close"]] % 5 != 0).any(axis=1)
    dias_fora = sorted({t.strftime("%d/%m") for t in d.index[fora]})
    print(f"   {k}: {nn(len(d))} barras, {d.index.normalize().nunique()} pregões, de {d.index[0]:%d/%m %H:%M} a"
          f" {d.index[-1]:%d/%m %H:%M}; preços fora do tick: {', '.join(dias_fora) if dias_fora else 'nenhum'}")
# Na série contínua, a plataforma multiplica os preços anteriores à rolagem por um fator; no WDO (rolagem no 1º dia útil do
# mês), os pregões de setembro saem com preços quebrados. Para não misturar preço ajustado com preço negociado,
# o WDO fica só com os pregões de outubro.
DADOS["WDO"] = DADOS["WDO"].loc["2026-10-01":]
print("   WDO: uso só os pregões de 01/10 em diante (antes, a série vem ajustada pela rolagem)")

## Parte 2: perfil de volume (Cap. 22)

In [ ]:
def perfil(d, passo):
    """Espalha os contratos de cada barra de 5 minutos por igual entre as faixas que a barra tocou."""
    vol = {}
    for o, h, l, c, v in d[["Open", "High", "Low", "Close", "Volume"]].to_numpy():
        faixas = np.arange(np.floor(l / passo) * passo, np.floor(h / passo) * passo + passo / 2, passo)
        for p in faixas:
            vol[p] = vol.get(p, 0.0) + v / len(faixas)
    return pd.Series(vol).sort_index()


def area_de_valor(s, frac=0.70):
    """POC e área de valor: parte do POC e soma, de cada lado, a faixa vizinha com mais volume até chegar a 70%."""
    i = lo = hi = int(np.argmax(s.to_numpy()))
    tot = s.iloc[i]
    while tot < frac * s.sum():
        cima = s.iloc[hi + 1] if hi + 1 < len(s) else -1
        baixo = s.iloc[lo - 1] if lo > 0 else -1
        if cima >= baixo:
            hi += 1
            tot += cima
        else:
            lo -= 1
            tot += baixo
    return s.index[i], s.index[lo], s.index[hi]


def resumo_dia(d, passo):
    s = perfil(d, passo)
    poc, val, vah = area_de_valor(s)
    tp = (d["High"] + d["Low"] + d["Close"]) / 3
    return {"abertura": d["Open"].iloc[0], "max": d["High"].max(), "min": d["Low"].min(), "fech": d["Close"].iloc[-1],
            "POC": poc + passo / 2, "VAL": val, "VAH": vah + passo, "VWAP": (tp * d["Volume"]).sum() / d["Volume"].sum()}


PERFIS = {}
print("\nParte 2. Perfil de volume de cada pregão (POC = preço com mais contratos; área de valor = 70% dos contratos)")
for k, d in DADOS.items():
    linhas = {dia: resumo_dia(g, PASSO[k]) for dia, g in d.groupby(d.index.date)}
    PERFIS[k] = pd.DataFrame(linhas).T
    PERFIS[k]["abriu"] = np.select([PERFIS[k]["abertura"] > PERFIS[k]["VAH"].shift(),
                                    PERFIS[k]["abertura"] < PERFIS[k]["VAL"].shift()], ["acima", "abaixo"], "dentro")
    PERFIS[k].loc[PERFIS[k].index[0], "abriu"] = ""
    k_ = 0 if k == "WIN" else 1
    print(f"   {k}   dia    abertura      POC   área de valor         VWAP  fechamento  abriu, contra a área de ontem")
    for dia, x in PERFIS[k].iterrows():
        print(f"         {dia:%d/%m} {br(x['abertura'], k_):>9} {br(x['POC'], k_):>8}  {br(x['VAL'], k_):>8} a"
              f" {br(x['VAH'], k_):<8} {br(x['VWAP'], k_):>9} {br(x['fech'], k_):>10}   {x['abriu']}")
for k in DADOS:
    p = PERFIS[k]
    ontem, hoje = p.loc[pd.Timestamp("2026-10-02").date()], p.loc[pd.Timestamp("2026-10-05").date()]
    lado = "acima" if hoje["abertura"] > ontem["VAH"] else "abaixo"
    volta = hoje["min"] <= ontem["VAH"] if lado == "acima" else hoje["max"] >= ontem["VAL"]
    print(f"   {k} em 05/10: abriu {lado} da área de valor de 02/10 ({br(ontem['VAL'], 0 if k == 'WIN' else 1)} a"
          f" {br(ontem['VAH'], 0 if k == 'WIN' else 1)}); voltou a ela no dia? {'sim' if volta else 'não'}")

In [ ]:
import matplotlib.pyplot as plt
k, dia = "WIN", "2026-10-05"
s = perfil(DADOS[k].loc[dia], PASSO[k])
poc, val, vah = area_de_valor(s)
ax = s.plot(kind="barh", figsize=(5, 8), color=["gray" if val <= p <= vah else "lightgray" for p in s.index], width=1)
ax.set_yticks(range(0, len(s), 10), [f"{p:,.0f}".replace(",", ".") for p in s.index[::10]])
ax.set_title(f"Perfil de volume do {k} em {dia} (cinza escuro: área de valor)")
plt.show()

## Parte 3: barras 1, 2, 3 e segundas entradas (Cap. 23)

In [ ]:
def cenario(d):
    """1 = barra de dentro; 2u / 2d = direcional para cima / para baixo; 3 = barra de fora (Caps. 23 e 24)."""
    h, l = d["High"], d["Low"]
    acima, abaixo = h > h.shift(), l < l.shift()
    c = pd.Series(np.select([acima & abaixo, acima, abaixo], ["3", "2u", "2d"], "1"), index=d.index)
    c.iloc[0] = ""
    return c


def resultado_r(h, l, c, i, entrada, stop, alvo_r=2.0, prazo=20):
    """Resultado em R de uma compra: alvo de alvo_r R, stop de 1R; se os dois na mesma barra, conta o stop."""
    risco = entrada - stop
    if risco <= 0:
        return np.nan
    alvo = entrada + alvo_r * risco
    fim = min(i + prazo, len(c) - 1)
    for k in range(i, fim + 1):
        if l[k] <= stop:
            return -1.0
        if h[k] >= alvo:
            return alvo_r
    return (c[fim] - entrada) / risco


def segundas_entradas(g):
    """H1 e H2 de Al Brooks no pregão, com tendência de alta = fechamento acima da média exponencial de 20 barras
    e média subindo. Devolve (posição da barra de entrada, tipo, entrada, stop, resultado em R)."""
    ema = g["Close"].ewm(span=20, adjust=False).mean()
    alta = ((g["Close"] > ema) & (ema > ema.shift(3))).to_numpy()
    H, L, C, O = (g[k].to_numpy() for k in ("High", "Low", "Close", "Open"))
    out, topo, recuo, conta, desceu = [], H[0], False, 0, False
    for i in range(1, len(H) - 1):
        if H[i] > topo and not recuo:
            topo = H[i]
            continue
        if not recuo and H[i] < H[i - 1]:
            recuo, conta, desceu = True, 0, True
            continue
        if recuo:
            if H[i] > H[i - 1] and desceu:
                conta += 1
                desceu = False
                if conta <= 2 and alta[i - 1]:                     # barra de sinal = a anterior
                    entrada = max(H[i - 1], O[i])                  # se abriu acima da ordem, compra na abertura
                    out.append((i, f"H{conta}", entrada, L[i - 1], resultado_r(H, L, C, i, entrada, L[i - 1])))
            elif H[i] <= H[i - 1]:
                desceu = True
            if H[i] > topo:
                topo, recuo = H[i], False
    return out


win = DADOS["WIN"]
cen = pd.concat([cenario(g) for _, g in win.groupby(win.index.date)])
cen = cen[cen != ""]
print("\nParte 3. WIN, 5 minutos: barras 1, 2 e 3 (a primeira barra de cada pregão fica de fora)")
for k, x in cen.replace({"2u": "2", "2d": "2"}).value_counts(normalize=True).sort_index().items():
    print(f"   barra {k}: {br(100 * x)}%")
SEG = []
for dia, g in win.groupby(win.index.date):
    for i, tipo, entrada, stop, r in segundas_entradas(g):
        SEG.append({"dia": dia, "hora": g.index[i], "tipo": tipo, "entrada": entrada, "stop": stop, "r": r})
SEG = pd.DataFrame(SEG)
print("   Segundas entradas de Brooks (alvo de 2R, stop de 1R na mínima da barra de sinal, até 20 barras, no pregão)")
for t, g in SEG.groupby("tipo"):
    print(f"   {t}: {len(g)} casos | acerto {br(100 * (g['r'] > 0).mean(), 0)}% | resultado médio {br(g['r'].mean(), 2)} R")
print("   Casos de 08/10/2026 (exemplo do Gráfico):")
for _, x in SEG[SEG["dia"] == pd.Timestamp("2026-10-08").date()].iterrows():
    print(f"      {x['hora']:%H:%M} {x['tipo']} compra em {nn(x['entrada'])}, stop {nn(x['stop'])}: {br(x['r'], 2)} R")

## Parte 4: The Strat no WDO de 60 minutos (Cap. 24)

In [ ]:
def agrega(d, regra="60min"):
    """Barras de 60 minutos que começam na hora cheia (9h, 10h, ...), pregão a pregão."""
    return d.resample(regra).agg({"Open": "first", "High": "max", "Low": "min", "Close": "last",
                                  "Volume": "sum"}).dropna()


def combos_strat(d):
    """2-1-2, 3-1-2 e 1-2-2 (RevStrat) nos dois sentidos. Gatilho: romper a máxima (ou mínima) da barra anterior;
    stop: o outro extremo da barra anterior; alvo: a máxima (ou mínima) da barra de duas atrás; sem alvo à frente, 1R."""
    c = cenario(d).to_numpy()
    O, H, L, C = (d[k].to_numpy() for k in ("Open", "High", "Low", "Close"))
    out = []
    for i in range(2, len(d)):
        a, b, x = c[i - 2], c[i - 1], c[i]
        nome = None
        if b == "1" and a in ("2u", "2d", "3") and x in ("2u", "2d"):
            nome = f"{'3' if a == '3' else '2'}-1-2"
        elif a == "1" and b in ("2u", "2d") and x in ("2u", "2d") and x != b:
            nome = "1-2-2 (RevStrat)"
        if not nome:
            continue
        compra = x == "2u"
        entrada = max(H[i - 1], O[i]) if compra else min(L[i - 1], O[i])
        stop = L[i - 1] if compra else H[i - 1]
        alvo = H[i - 2] if compra else L[i - 2]
        if (alvo - entrada) * (1 if compra else -1) <= 0:
            alvo = entrada + (entrada - stop)
        risco = abs(entrada - stop)
        res = np.nan
        for k in range(i, len(d)):
            bate_stop = L[k] <= stop if compra else H[k] >= stop
            bate_alvo = H[k] >= alvo if compra else L[k] <= alvo
            if bate_stop and (k > i or not bate_alvo):
                res = -1.0
                break
            if bate_alvo:
                res = abs(alvo - entrada) / risco
                break
        out.append({"hora": d.index[i], "combo": nome, "sentido": "alta" if compra else "baixa",
                    "sequencia": f"{a}-{b}-{x}", "entrada": entrada, "stop": stop, "alvo": alvo, "r": res})
    return pd.DataFrame(out)


H60 = agrega(DADOS["WDO"])
H60["cenario"] = cenario(H60)
COMBOS = combos_strat(H60)
print("\nParte 4. WDO, 60 minutos (01 a 06/10/2026): cenário de cada barra")
for dia, g in H60.groupby(H60.index.date):
    print(f"   {dia:%d/%m}: " + " ".join(f"{t:%H}h={x or '-'}" for t, x in g["cenario"].items()))
print("   Combos encontrados (a barra do gatilho cruza pregões: a hora é a da barra que dispara)")
for _, x in COMBOS.iterrows():
    print(f"   {x['hora']:%d/%m %Hh} {x['combo']:<17} de {x['sentido']:<5} ({x['sequencia']}): entrada {br(x['entrada'])},"
          f" stop {br(x['stop'])}, alvo {br(x['alvo'])} -> {br(x['r'], 2) if x['r'] == x['r'] else 'em aberto'} R")
dia_ = agrega(DADOS["WDO"], "1D")
print("   Diário do WDO (para a continuidade temporal): " +
      " ".join(f"{t:%d/%m}={'alta' if c_ >= o_ else 'baixa'}" for t, o_, c_ in zip(dia_.index, dia_.Open, dia_.Close)))

## Parte 5: pivôs clássicos do dia (Cap. 25)

In [ ]:
def pivos(h, l, c):
    p = (h + l + c) / 3
    return {"R2": p + (h - l), "R1": 2 * p - l, "P": p, "S1": 2 * p - h, "S2": p - (h - l)}


PIVOS = {}
print("\nParte 5. Pivôs clássicos do dia, com máxima, mínima e último preço do pregão anterior (não o ajuste da B3)")
for k in DADOS:
    p = PERFIS[k]
    lin = []
    for j in range(1, len(p)):
        nv = pivos(p["max"].iloc[j - 1], p["min"].iloc[j - 1], p["fech"].iloc[j - 1])
        hoje = p.iloc[j]
        lin.append({"dia": p.index[j], **nv, "max": hoje["max"], "min": hoje["min"], "fech": hoje["fech"],
                    "tocou_S1": hoje["min"] <= nv["S1"], "tocou_R1": hoje["max"] >= nv["R1"],
                    "fechou_acima_S1": hoje["fech"] > nv["S1"], "fechou_abaixo_R1": hoje["fech"] < nv["R1"]})
    PIVOS[k] = pd.DataFrame(lin).set_index("dia")
    t = PIVOS[k]
    s1, r1 = t["tocou_S1"], t["tocou_R1"]
    print(f"   {k}: {len(t)} pregões | tocou o S1 em {s1.sum()} e fechou acima dele em {(s1 & t['fechou_acima_S1']).sum()}"
          f" | tocou o R1 em {r1.sum()} e fechou abaixo dele em {(r1 & t['fechou_abaixo_R1']).sum()}")
for k in DADOS:
    x = PIVOS[k].loc[pd.Timestamp("2026-10-06").date()]
    k_ = 0 if k == "WIN" else 1
    print(f"   {k} em 06/10: " + " ".join(f"{n} {br(x[n], k_)}" for n in ("S2", "S1", "P", "R1", "R2")) +
          f" | mínima {br(x['min'], k_)}, máxima {br(x['max'], k_)}, último {br(x['fech'], k_)}")

## Parte 6: o Market Profile testado em 196 pregões (Cap. 22)

In [ ]:
LONGOS = {k: carrega(os.path.join(PASTA, a)) for k, a in LONGO.items()}


def dias(d, passo):
    """Resumo de cada pregão: abertura, máxima, mínima, fechamento, POC, área de valor, VWAP e equilíbrio inicial
    (a primeira hora, contada da primeira barra do dia: em 18/02, quarta-feira de cinzas, o pregão abriu às 13h)."""
    linhas = {}
    for dia, g in d.groupby(d.index.date):
        r = resumo_dia(g, passo)
        ei = g[g.index < g.index[0] + pd.Timedelta("60min")]
        r.update({"EIH": ei["High"].max(), "EIL": ei["Low"].min()})
        linhas[dia] = r
    return pd.DataFrame(linhas).T


def opera(g, t0, lado, entrada, stop, alvo, custo=0.0, alvo_na_barra=True):
    """Resultado em R de uma operação no pregão g a partir da barra t0 (lado 1 = compra, -1 = venda). Stop antes do
    alvo na mesma barra; sem alvo nem stop, sai no último preço do dia. custo: em pontos, ida e volta."""
    risco = abs(entrada - stop)
    for n, (_, b) in enumerate(g.loc[t0:].iterrows()):
        if (b["Low"] <= stop) if lado == 1 else (b["High"] >= stop):
            return -1.0 - custo / risco
        if (n > 0 or alvo_na_barra) and ((b["High"] >= alvo) if lado == 1 else (b["Low"] <= alvo)):
            return abs(alvo - entrada) / risco - custo / risco
    return (g["Close"].iloc[-1] - entrada) * lado / risco - custo / risco


TESTE22 = {}
print("\nParte 6. Market Profile em 196 pregões (02/01 a 14/10/2026, série contínua ajustada)")
for k, d in LONGOS.items():
    P = dias(d, PASSO[k])
    ont = P.shift()
    P["lado"] = np.select([P["abertura"] > ont["VAH"], P["abertura"] < ont["VAL"]], ["acima", "abaixo"], "dentro")
    P = P.iloc[1:]
    P["dia_ret"] = P["fech"] / P["abertura"] - 1
    t = {}
    print(f"   {k}: {len(P)} pregões")
    for lado in ("acima", "dentro", "abaixo"):
        g = P[P["lado"] == lado]
        t[lado] = (len(g), (g["dia_ret"] > 0).mean(), g["dia_ret"].mean())
        print(f"      abriu {lado:<6} da área de valor de ontem: {len(g):3d} dias | fechou acima da abertura em"
              f" {br(100 * t[lado][1], 0)}% | dia médio {br(100 * t[lado][2], 2)}%")
    # regra dos 80%: abriu fora, dois fechamentos seguidos de 30 minutos dentro da área de ontem -> atravessa a área?
    c80 = ok80 = cc = okc = 0
    for dia in P.index[P["lado"] != "dentro"]:
        g, y = d.loc[str(dia)], ont.loc[dia]
        cima = P.loc[dia, "lado"] == "acima"
        alvo = y["VAL"] if cima else y["VAH"]
        atravessa = (lambda r: (r["Low"] <= alvo).any()) if cima else (lambda r: (r["High"] >= alvo).any())
        m30 = g.resample("30min").agg({"High": "max", "Low": "min", "Close": "last"}).dropna()
        dentro = (m30["Close"] <= y["VAH"]) & (m30["Close"] >= y["VAL"])
        dois = dentro & dentro.shift(1, fill_value=False)
        if dois.any():
            c80 += 1
            ok80 += atravessa(g.loc[dois.idxmax() + pd.Timedelta("30min"):])
        toca = (g["Low"] <= y["VAH"]) & (g["High"] >= y["VAL"])
        if toca.any():
            cc += 1
            okc += atravessa(g.loc[toca.idxmax():])
    t["80"] = (c80, ok80 / max(c80, 1), cc, okc / max(cc, 1))
    print(f"      regra dos 80% (abriu fora e fechou dois períodos de 30 min dentro): {c80} casos, atravessou a área em"
          f" {br(100 * t['80'][1], 0)}% | só encostou na área: {cc} casos, atravessou em {br(100 * t['80'][3], 0)}%")
    # POC virgem: o POC que o pregão seguinte não tocou volta a ser tocado nos 10 pregões depois?
    tocou = (P["min"].shift(-1) <= P["POC"]) & (P["max"].shift(-1) >= P["POC"])
    hit, ctl = [], []
    for j in range(len(P) - 11):
        if tocou.iloc[j]:
            continue
        poc, nxt, fut = P["POC"].iloc[j], P.iloc[j + 1], P.iloc[j + 2:j + 12]
        espelho = 2 * nxt["fech"] - poc            # nível à mesma distância do fechamento, do outro lado
        hit.append(((fut["min"] <= poc) & (fut["max"] >= poc)).any())
        ctl.append(((fut["min"] <= espelho) & (fut["max"] >= espelho)).any())
    t["virgem"] = (len(hit), np.mean(hit), np.mean(ctl))
    print(f"      POC virgem: {len(hit)} casos; tocado nos 10 pregões seguintes em {br(100 * np.mean(hit), 0)}% | nível"
          f" de controle: {br(100 * np.mean(ctl), 0)}%")
    # equilíbrio inicial: rompido? e a operação de rompimento (stop no outro lado, alvo de uma vez a altura do EI)
    rompe = (P["max"] > P["EIH"]) | (P["min"] < P["EIL"])
    res_ei = []
    for dia in P.index:
        g = d.loc[str(dia)]
        H, L_ = P.loc[dia, "EIH"], P.loc[dia, "EIL"]
        dep = g[g.index >= g.index[0] + pd.Timedelta("60min")]
        cima, baixo = dep.index[dep["High"] > H], dep.index[dep["Low"] < L_]
        if not len(cima) and not len(baixo):
            continue
        if len(cima) and (not len(baixo) or cima[0] <= baixo[0]):
            res_ei.append(opera(g, cima[0], 1, H, L_, H + (H - L_), 2 * TICK[k]))
        else:
            res_ei.append(opera(g, baixo[0], -1, L_, H, L_ - (H - L_), 2 * TICK[k]))
    res_ei = np.array(res_ei)
    larg = ((P["EIH"] - P["EIL"]) / (P["max"] - P["min"])).median()
    t["EI"] = (rompe.mean(), larg, len(res_ei), (res_ei > 0).mean(), res_ei.mean())
    print(f"      equilíbrio inicial: rompido em {br(100 * rompe.mean(), 0)}% dos dias; a primeira hora fez {br(100 * larg, 0)}%"
          f" da amplitude do dia (mediana); operar o rompimento: {len(res_ei)} operações, acerto"
          f" {br(100 * (res_ei > 0).mean(), 0)}%, {br(res_ei.mean(), 2)} R com custo de 1 tick por lado")
    dist = ((P["POC"] - P["VWAP"]).abs() / P["fech"]).median()
    t["pocvwap"] = dist
    print(f"      POC x VWAP: distância mediana de {br(100 * dist, 2)}% do preço")
    TESTE22[k] = t

## Parte 7: os pivôs do dia testados em 196 pregões (Cap. 25)

In [ ]:
TESTE25 = {}
print("\nParte 7. Pivôs do dia em 196 pregões. Nível falso: os pivôs de 7 pregões antes, na escala de ontem")
for k, d in LONGOS.items():
    P = dias(d, PASSO[k])
    toques = {n: [] for n in ("R2", "R1", "P", "S1", "S2")}
    ops = {"pivôs de ontem": [], "nível falso": []}
    for j in range(7, len(P)):
        dia, y, z = P.index[j], P.iloc[j - 1], P.iloc[j - 8]
        g, hoje = d.loc[str(dia)], P.iloc[j]
        nv = pivos(y["max"], y["min"], y["fech"])
        esc = y["fech"] / z["fech"]
        nf = {n: v * esc for n, v in pivos(z["max"], z["min"], z["fech"]).items()}
        for n in toques:
            toques[n].append((hoje["min"] <= nv[n]) & (hoje["max"] >= nv[n]))
        # revés nos extremos: vende no primeiro toque do R2 (alvo R1) ou compra no primeiro toque do S2 (alvo S1),
        # stop à mesma distância do outro lado; ordem limitada no nível; o alvo só vale a partir da barra seguinte
        for nome, v in (("pivôs de ontem", nv), ("nível falso", nf)):
            o = g["Open"].iloc[0]
            tr, ts = g.index[g["High"] >= v["R2"]], g.index[g["Low"] <= v["S2"]]
            if len(tr) and o < v["R2"] and (not len(ts) or tr[0] <= ts[0]):
                r = opera(g, tr[0], -1, v["R2"], 2 * v["R2"] - v["R1"], v["R1"], 2 * TICK[k], alvo_na_barra=False)
            elif len(ts) and o > v["S2"]:
                r = opera(g, ts[0], 1, v["S2"], 2 * v["S2"] - v["S1"], v["S1"], 2 * TICK[k], alvo_na_barra=False)
            else:
                continue
            ops[nome].append((pd.Timestamp(dia), r))
    t = {"toques": {n: np.mean(v) for n, v in toques.items()}, "ops": {}}
    print(f"   {k}: pregões que tocaram cada nível: " +
          " | ".join(f"{n} {br(100 * v, 0)}%" for n, v in t["toques"].items()))
    for nome, lst in ops.items():
        s = pd.Series(dict(lst))
        a, b = s[:"2026-05-31"], s["2026-06-01":]
        t["ops"][nome] = (len(s), (s > 0).mean(), s.mean(), s.std() / np.sqrt(len(s)), a.mean(), b.mean())
        print(f"      revés no R2/S2, {nome:<15}: {len(s):3d} operações | acerto {br(100 * (s > 0).mean(), 0)}% |"
              f" {br(s.mean(), 2)} R (erro padrão {br(s.std() / np.sqrt(len(s)), 2)}) | jan. a maio {br(a.mean(), 2)} R,"
              f" jun. a out. {br(b.mean(), 2)} R")
    TESTE25[k] = t
print("   Custo: 1 tick por lado. Amostra curta: o erro padrão mostra o tamanho da incerteza (ver o Aviso legal).")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `perfil`, `area_de_valor` e `resumo_dia` da Parte 2. Mostre onde os contratos de cada barra de 5 minutos são espalhados por igual entre as faixas de tamanho `PASSO`, por que o POC é a faixa com mais contratos (a moda) e por que a VWAP é uma média." Confira com a Parte 2, que imprime o perfil de cada pregão: no WIN, em 05/10/2026, POC em 209.625, área de valor de 207.900 a 210.000 e VWAP em 208.640.
2. **Adaptar:** "Numa célula nova depois da Parte 4, leia com `carrega` um CSV de 5 minutos de [ATIVO], exportado da sua plataforma no mesmo formato dos arquivos da pasta `dados_futuros`. Se for uma série contínua, use só os pregões depois da última rolagem, como a Parte 1 faz com o WDO. Agregue em 60 minutos com `agrega` e rode `combos_strat`. Não mude o gatilho, o stop nem o alvo." Antes de usar o seu arquivo, rode o notebook como está e confira se a Parte 4 mostra o 3-1-2 de baixa de 02/10 no WDO, com venda em 5.249,0, stop em 5.259,0 e alvo em 5.244,5, como no livro. Poucos combos não são amostra: qualquer conclusão precisa do teste fora da amostra (Capítulo 27).

Mais pedidos estão nas páginas dos Capítulos 22 a 25, no hub do livro.

## Para ir além
1. Na Parte 2, troque o PASSO do WIN de 50 para 25 pontos. O POC e a área de valor mudam muito?
2. Exporte da sua plataforma um mês inteiro de WIN e rode de novo a Parte 3. O H2 continua pior que o H1?
3. Na Parte 4, troque as barras de 60 para 30 minutos (agrega(DADOS['WDO'], '30min')). Quantos combos aparecem, e quantos chegam ao alvo?
4. Na Parte 5, calcule os pivôs com o preço de ajuste da B3 no lugar do último preço. Os níveis mudam?
5. Na Parte 7, troque o alvo do revés para o P em vez do R1/S1. O resultado melhora? Lembre: cada teste a mais aumenta a chance de achar sorte (Capítulo 27).

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.